<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/15B_NeuroFHIR_Review_WISH_Live_Deployment_QA_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-Review — Notebook 15B
## Live GitHub Pages Post-Deployment QA

This notebook checks the **publicly deployed** NeuroFHIR-Review app after Notebook 15A.

### URLs
- Existing NeuroFHIR-QC / AMIA app: `https://sanghati23.github.io/neurofhir-qc/`
- New frozen study app: `https://sanghati23.github.io/neurofhir-qc/wish-review/`

### What this notebook certifies
1. Both public URLs return HTTP 200.
2. `/wish-review/` is a distinct application and not an accidental copy of the root page.
3. The public deployment manifest is reachable.
4. Every frozen participant file listed in the manifest is downloadable and has the **same SHA-256 hash** as the frozen source recorded at deployment time.
5. No researcher-only filenames or forbidden researcher fields are publicly exposed in the deployed participant package.
6. The rendered page loads in a real headless Chromium browser.
7. The rendered UI identifies itself as NeuroFHIR-Review and contains the expected study-stage language.
8. JavaScript console errors, page exceptions, and failed network requests are captured.
9. No unexpected POST/PUT/PATCH/DELETE requests are made during initial page load.
10. A QA screenshot is produced for visual confirmation.

This is a **deployment QA notebook only**. It must not be used to create study data.

In [1]:
# Cell 1 — Configuration
from pathlib import Path

ROOT_URL = "https://sanghati23.github.io/neurofhir-qc/"
WISH_URL = ROOT_URL + "wish-review/"
MANIFEST_URL = WISH_URL + "deployment_manifest.json"

QA_OUT = Path("/content/neurofhir_review_live_qa")
QA_OUT.mkdir(parents=True, exist_ok=True)

print("Root app:", ROOT_URL)
print("WISH app:", WISH_URL)
print("Manifest:", MANIFEST_URL)

Root app: https://sanghati23.github.io/neurofhir-qc/
WISH app: https://sanghati23.github.io/neurofhir-qc/wish-review/
Manifest: https://sanghati23.github.io/neurofhir-qc/wish-review/deployment_manifest.json


In [2]:
# Cell 2 — HTTP reachability
import requests, hashlib, json, re, time
from urllib.parse import urljoin

session = requests.Session()
session.headers.update({
    "User-Agent": "NeuroFHIR-Review-Deployment-QA/1.0",
    "Cache-Control": "no-cache",
})

def get(url):
    r = session.get(url, timeout=30)
    print(r.status_code, r.url, r.headers.get("content-type", ""))
    return r

root_r = get(ROOT_URL)
wish_r = get(WISH_URL)
manifest_r = get(MANIFEST_URL)

assert root_r.status_code == 200, f"Root app HTTP {root_r.status_code}"
assert wish_r.status_code == 200, f"WISH app HTTP {wish_r.status_code}"
assert manifest_r.status_code == 200, f"Manifest HTTP {manifest_r.status_code}"

print("✅ HTTP reachability: PASS")

200 https://sanghati23.github.io/neurofhir-qc/ text/html; charset=utf-8
200 https://sanghati23.github.io/neurofhir-qc/wish-review/ text/html; charset=utf-8
200 https://sanghati23.github.io/neurofhir-qc/wish-review/deployment_manifest.json application/json; charset=utf-8
✅ HTTP reachability: PASS


In [3]:
# Cell 3 — Manifest validation
manifest = manifest_r.json()

required_manifest_fields = [
    "artifact",
    "deployment_subpath",
    "target_url",
    "source_file_count",
    "source_sha256",
    "researcher_only_deployed",
    "qa_participant_ids_reserved",
    "recommended_first_real_participant_id",
]

missing = [k for k in required_manifest_fields if k not in manifest]
assert not missing, f"Manifest missing fields: {missing}"

assert manifest["deployment_subpath"] == "/wish-review/"
assert manifest["researcher_only_deployed"] is False
assert manifest["qa_participant_ids_reserved"] == ["P001", "P002"]
assert manifest["recommended_first_real_participant_id"] == "P003"
assert isinstance(manifest["source_sha256"], dict) and manifest["source_sha256"]
assert manifest["source_file_count"] == len(manifest["source_sha256"])

print("✅ Deployment manifest: PASS")
print("Artifact:", manifest["artifact"])
print("Frozen file count:", manifest["source_file_count"])
print("First real participant ID:", manifest["recommended_first_real_participant_id"])

✅ Deployment manifest: PASS
Artifact: NeuroFHIR-Review frozen participant application
Frozen file count: 14
First real participant ID: P003


In [4]:
# Cell 4 — Verify every live frozen file byte-for-byte by SHA-256
def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

hash_failures = []
download_failures = []

for rel, expected_hash in sorted(manifest["source_sha256"].items()):
    url = urljoin(WISH_URL, rel)
    try:
        r = session.get(url, timeout=30)
        if r.status_code != 200:
            download_failures.append((rel, r.status_code))
            continue
        actual = sha256_bytes(r.content)
        if actual != expected_hash:
            hash_failures.append((rel, expected_hash, actual))
    except Exception as e:
        download_failures.append((rel, repr(e)))

assert not download_failures, "Live frozen files missing/unreachable: " + repr(download_failures[:20])
assert not hash_failures, "Live file hash mismatch: " + repr(hash_failures[:20])

print(f"✅ Live byte-for-byte integrity: PASS ({len(manifest['source_sha256'])} files)")

✅ Live byte-for-byte integrity: PASS (14 files)


In [5]:
# Cell 5 — Root app and WISH app must be distinct
root_hash = sha256_bytes(root_r.content)
wish_hash = sha256_bytes(wish_r.content)

assert root_hash != wish_hash, (
    "Root page and /wish-review/ have identical HTML. "
    "This suggests the separate app may not actually be deployed."
)

wish_text = wish_r.text.lower()
assert "neurofhir-review" in wish_text or "neurofhir review" in wish_text, (
    "Public /wish-review/ HTML does not identify itself as NeuroFHIR-Review."
)

print("✅ Separate-route identity: PASS")
print("Root index SHA-256:", root_hash)
print("WISH index SHA-256:", wish_hash)

✅ Separate-route identity: PASS
Root index SHA-256: 443e5563f144837783946a163c25c968719455d1d3840bd20d34b53f2b90eca3
WISH index SHA-256: 0600d1ad3fb6706fcc322dc6407d4b8ab54a3acf7cd176b2b833178bc604c4c3


In [6]:
# Cell 6 — Public leakage scan across every frozen participant file
FORBIDDEN_TEXT_MARKERS = [
    "source_case_id",
    "ai_correctness",
    "reference_workflow_disposition_path_b",
    "path_a_reference_status",
    "final_researcher_scenario_key",
    "researcher_only",
    "researcher trajectory",
    "reference_volume",
    "reference volume",
]

FORBIDDEN_FILENAME_MARKERS = [
    "researcher",
    "answer_key",
    "scenario_key",
    "reference_workflow",
]

TEXT_SUFFIXES = {
    ".html", ".htm", ".js", ".mjs", ".cjs", ".json",
    ".css", ".txt", ".csv", ".md", ".xml"
}

leaks = []

for rel in sorted(manifest["source_sha256"]):
    rel_lower = rel.lower()

    for marker in FORBIDDEN_FILENAME_MARKERS:
        if marker in rel_lower:
            leaks.append(f"filename:{rel} -> {marker}")

    suffix = Path(rel).suffix.lower()
    if suffix in TEXT_SUFFIXES:
        r = session.get(urljoin(WISH_URL, rel), timeout=30)
        txt = r.text.lower()
        for marker in FORBIDDEN_TEXT_MARKERS:
            if marker in txt:
                leaks.append(f"content:{rel} -> {marker}")

assert not leaks, (
    "❌ PUBLIC RESEARCHER-ONLY LEAKAGE DETECTED:\n" + "\n".join(leaks[:50])
)

print("✅ Public participant/researcher separation: PASS")

✅ Public participant/researcher separation: PASS


In [7]:
# Cell 7 — Install Playwright + ALL Chromium system dependencies in Colab
#
# IMPORTANT:
# The previous version only ran `playwright install chromium`.
# That downloads Chromium but does NOT guarantee Linux libraries such as
# libatk-1.0.so.0 are present. This cell installs both the browser and its
# operating-system dependencies, then checks the executable with `ldd`.

import subprocess, sys
from pathlib import Path

print("Installing/updating Playwright...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "playwright"],
    check=True,
)

print("Installing Chromium Linux dependencies...")
deps = subprocess.run(
    ["playwright", "install-deps", "chromium"],
    text=True,
    capture_output=True,
)
if deps.returncode != 0:
    print(deps.stdout[-4000:])
    print(deps.stderr[-4000:])
    raise RuntimeError("Playwright Chromium system-dependency installation failed.")

print("Installing Playwright Chromium...")
browser_install = subprocess.run(
    ["playwright", "install", "chromium"],
    text=True,
    capture_output=True,
)
if browser_install.returncode != 0:
    print(browser_install.stdout[-4000:])
    print(browser_install.stderr[-4000:])
    raise RuntimeError("Playwright Chromium installation failed.")

# Find the installed headless shell and verify all shared libraries resolve.
cache = Path.home() / ".cache" / "ms-playwright"
candidates = sorted(
    list(cache.glob("chromium_headless_shell-*/chrome-headless-shell-linux64/chrome-headless-shell"))
    + list(cache.glob("chromium-*/chrome-linux/chrome"))
    + list(cache.glob("chromium-*/chrome-linux64/chrome"))
)

assert candidates, f"No Playwright Chromium executable found under {cache}"

chromium_exe = candidates[-1]
print("Chromium executable:", chromium_exe)

ldd = subprocess.run(
    ["ldd", str(chromium_exe)],
    text=True,
    capture_output=True,
    check=True,
).stdout

missing_libs = [
    line.strip()
    for line in ldd.splitlines()
    if "not found" in line.lower()
]

assert not missing_libs, (
    "Chromium still has unresolved Linux libraries:\n"
    + "\n".join(missing_libs)
)

print("✅ Chromium dependency check: PASS")
print("✅ libatk / GTK / NSS / X11 runtime dependencies resolved.")

Installing/updating Playwright...
Installing Chromium Linux dependencies...
Installing Playwright Chromium...
Chromium executable: /root/.cache/ms-playwright/chromium_headless_shell-1234/chrome-headless-shell-linux64/chrome-headless-shell
✅ Chromium dependency check: PASS
✅ libatk / GTK / NSS / X11 runtime dependencies resolved.


In [8]:
# Cell 8 — Real Chromium render, console/network QA, screenshot
import asyncio, json, os, textwrap, traceback
from playwright.async_api import async_playwright

async def browser_qa():
    console_errors = []
    page_errors = []
    failed_requests = []
    write_requests = []
    observed_requests = []

    async with async_playwright() as p:
        browser = None
        try:
            browser = await p.chromium.launch(
                headless=True,
                args=[
                    "--no-sandbox",
                    "--disable-setuid-sandbox",
                    "--disable-dev-shm-usage",
                    "--disable-gpu",
                ],
            )

            context = await browser.new_context(
                viewport={"width": 1440, "height": 1100},
                ignore_https_errors=False,
            )
            page = await context.new_page()

            def on_console(msg):
                if msg.type == "error":
                    console_errors.append(msg.text)

            def on_pageerror(exc):
                page_errors.append(str(exc))

            def on_requestfailed(req):
                failed_requests.append({
                    "url": req.url,
                    "method": req.method,
                    "failure": req.failure,
                })

            def on_request(req):
                observed_requests.append({
                    "url": req.url,
                    "method": req.method,
                })
                if req.method.upper() in {"POST", "PUT", "PATCH", "DELETE"}:
                    write_requests.append({
                        "url": req.url,
                        "method": req.method,
                    })

            page.on("console", on_console)
            page.on("pageerror", on_pageerror)
            page.on("requestfailed", on_requestfailed)
            page.on("request", on_request)

            response = await page.goto(
                WISH_URL,
                wait_until="networkidle",
                timeout=90000,
            )

            assert response is not None
            assert response.status == 200, f"Browser navigation HTTP {response.status}"

            # Give MutationObserver/UI-shell code a moment to settle.
            await page.wait_for_timeout(1200)

            title = await page.title()
            body_text = await page.locator("body").inner_text()

            screenshot_path = QA_OUT / "wish_review_live_fullpage.png"
            await page.screenshot(
                path=str(screenshot_path),
                full_page=True,
            )

            # Structural checks specific to the 15A2 Evidence Cockpit.
            cockpit = {
                "left_rail": await page.locator("#nf-cockpit-left").count(),
                "top_bar": await page.locator("#nf-cockpit-top").count(),
                "right_console": await page.locator("#nf-cockpit-right").count(),
                "case_dots": await page.locator(".nf-case-dot").count(),
            }

            buttons = await page.locator("button").all_inner_texts()
            inputs = await page.locator("input").count()
            selects = await page.locator("select").count()
            textareas = await page.locator("textarea").count()

            result = {
                "title": title,
                "body_text": body_text,
                "buttons": buttons,
                "inputs": inputs,
                "selects": selects,
                "textareas": textareas,
                "console_errors": console_errors,
                "page_errors": page_errors,
                "failed_requests": failed_requests,
                "write_requests": write_requests,
                "request_count": len(observed_requests),
                "screenshot": str(screenshot_path),
                "cockpit": cockpit,
            }

            return result

        finally:
            if browser is not None:
                await browser.close()

browser_result = await browser_qa()

print("Page title:", browser_result["title"])
print("Evidence Cockpit:", browser_result["cockpit"])
print("Buttons:", browser_result["buttons"])
print("Inputs:", browser_result["inputs"])
print("Selects:", browser_result["selects"])
print("Textareas:", browser_result["textareas"])
print("Network requests:", browser_result["request_count"])
print("Console errors:", browser_result["console_errors"])
print("Page errors:", browser_result["page_errors"])
print("Failed requests:", browser_result["failed_requests"])
print("Unexpected write requests:", browser_result["write_requests"])
print("Screenshot:", browser_result["screenshot"])

assert browser_result["cockpit"]["left_rail"] == 1
assert browser_result["cockpit"]["top_bar"] == 1
assert browser_result["cockpit"]["right_console"] == 1
assert browser_result["cockpit"]["case_dots"] == 12

print("✅ Chromium rendered the live 15A2 Evidence Cockpit successfully.")

Page title: NeuroFHIR-Review Study
Evidence Cockpit: {'left_rail': 1, 'top_bar': 1, 'right_console': 1, 'case_dots': 12}
Buttons: ['Start review']
Inputs: 1
Selects: 1
Textareas: 0
Network requests: 2
Console errors: []
Page errors: []
Failed requests: []
Unexpected write requests: []
Screenshot: /content/neurofhir_review_live_qa/wish_review_live_fullpage.png
✅ Chromium rendered the live 15A2 Evidence Cockpit successfully.


In [9]:
# Cell 9 — Rendered UI + progressive-disclosure gate
#
# IMPORTANT:
# NeuroFHIR-Review intentionally uses progressive disclosure.
# On initial page load, later study stages such as "Initial judgment"
# and "Final action" may be HIDDEN by the frozen protocol.
#
# Therefore:
#   - visible-page checks validate only what should exist NOW;
#   - later-stage markers are checked in the deployed HTML/source,
#     NOT required to be visible simultaneously.

body_lower = browser_result["body_text"].lower()
html_lower = wish_r.text.lower()

# A) What MUST be visible on the initial rendered screen.
visible_expected = {
    "NeuroFHIR-Review identity": [
        "neurofhir-review",
        "neurofhir review",
    ],
    "review workflow context": [
        "case",
        "review",
    ],
}

visible_missing = []
for label, alternatives in visible_expected.items():
    if not any(term in body_lower for term in alternatives):
        visible_missing.append(label)

assert not visible_missing, (
    "Rendered initial UI missing expected visible markers: "
    + ", ".join(visible_missing)
)

# B) 15A2 Evidence Cockpit structure must be rendered.
cockpit = browser_result.get("cockpit", {})
assert cockpit.get("left_rail") == 1, "Left Evidence Cockpit rail not rendered."
assert cockpit.get("top_bar") == 1, "Top Evidence Cockpit context bar not rendered."
assert cockpit.get("right_console") == 1, "Right Evidence Cockpit console not rendered."
assert cockpit.get("case_dots") == 12, "Expected 12 case-progress cells."

# C) Later-stage study functionality must exist in the deployed source,
# but should NOT be required to be visible on the first screen.
source_expected_groups = {
    "initial judgment stage": [
        "initial judgment",
        "initial_judgment",
    ],
    "final action stage": [
        "final action",
        "final_action",
        "final disposition",
    ],
    "confidence capture": [
        "initial_confidence",
        "final_confidence",
        "confidence",
    ],
    "rationale capture": [
        "rationale",
        "reason_code",
        "reason code",
    ],
}

source_missing = []
for label, alternatives in source_expected_groups.items():
    if not any(term in html_lower for term in alternatives):
        source_missing.append(label)

assert not source_missing, (
    "Deployed source is missing expected later-stage study functionality: "
    + ", ".join(source_missing)
)

# D) Browser/runtime health.
assert not browser_result["page_errors"], (
    "JavaScript page exceptions detected: "
    + repr(browser_result["page_errors"])
)

same_origin_failures = [
    x for x in browser_result["failed_requests"]
    if x["url"].startswith(ROOT_URL)
]

assert not same_origin_failures, (
    "Failed same-origin requests detected: "
    + repr(same_origin_failures)
)

assert not browser_result["write_requests"], (
    "Unexpected network write requests occurred during initial load: "
    + repr(browser_result["write_requests"])
)

print("✅ Initial rendered UI identity: PASS")
print("✅ 15A2 Evidence Cockpit structure: PASS")
print("✅ Progressive disclosure respected: PASS")
print("   - Initial judgment exists in deployed source; not required on first screen.")
print("   - Final action exists in deployed source; not required on first screen.")
print("✅ JavaScript page exceptions: NONE")
print("✅ Same-origin failed resources: NONE")
print("✅ Unexpected network writes on initial load: NONE")

✅ Initial rendered UI identity: PASS
✅ 15A2 Evidence Cockpit structure: PASS
✅ Progressive disclosure respected: PASS
   - Initial judgment exists in deployed source; not required on first screen.
   - Final action exists in deployed source; not required on first screen.
✅ JavaScript page exceptions: NONE
✅ Same-origin failed resources: NONE
✅ Unexpected network writes on initial load: NONE


In [10]:
# Cell 10 — Save machine-readable QA report
report = {
    "root_url": ROOT_URL,
    "wish_url": WISH_URL,
    "manifest_url": MANIFEST_URL,
    "http": {
        "root_status": root_r.status_code,
        "wish_status": wish_r.status_code,
        "manifest_status": manifest_r.status_code,
    },
    "manifest": {
        "artifact": manifest.get("artifact"),
        "source_file_count": manifest.get("source_file_count"),
        "researcher_only_deployed": manifest.get("researcher_only_deployed"),
        "qa_participant_ids_reserved": manifest.get("qa_participant_ids_reserved"),
        "recommended_first_real_participant_id": manifest.get("recommended_first_real_participant_id"),
    },
    "integrity": {
        "all_live_hashes_match": True,
        "public_leakage_detected": False,
        "root_and_wish_distinct": root_hash != wish_hash,
    },
    "browser": {
        "title": browser_result["title"],
        "buttons": browser_result["buttons"],
        "inputs": browser_result["inputs"],
        "selects": browser_result["selects"],
        "textareas": browser_result["textareas"],
        "console_errors": browser_result["console_errors"],
        "page_errors": browser_result["page_errors"],
        "failed_requests": browser_result["failed_requests"],
        "write_requests": browser_result["write_requests"],
        "screenshot": browser_result["screenshot"],
    },
}

report_path = QA_OUT / "notebook_15b_live_deployment_qa.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("✅ QA report written:", report_path)

✅ QA report written: /content/neurofhir_review_live_qa/notebook_15b_live_deployment_qa.json


In [11]:
# Cell 11 — Final live deployment gate
print("=" * 78)
print("NOTEBOOK 15B — LIVE PUBLIC DEPLOYMENT QA")
print("=" * 78)
print("Root NeuroFHIR-QC URL HTTP 200:               PASS")
print("Separate /wish-review/ URL HTTP 200:          PASS")
print("Deployment manifest reachable:                PASS")
print("Frozen public files hash-match deployment:    PASS")
print("Root app and WISH app are distinct:            PASS")
print("Researcher-only leakage scan:                  PASS")
print("Rendered Chromium load:                        PASS")
print("Progressive-disclosure UI/source markers:      PASS")
print("JavaScript page exceptions:                    PASS")
print("Failed same-origin resources:                  PASS")
print("Unexpected network writes on initial load:     PASS")
print("=" * 78)
print("✅ NOTEBOOK 15B LIVE DEPLOYMENT GATE: TRUE")
print()
print("Public participant app:")
print(WISH_URL)
print()
print("IMPORTANT:")
print("- P001/P002 remain QA-only IDs.")
print("- Begin real participant allocation with P003.")
print("- Do not use this QA notebook to generate study responses.")

NOTEBOOK 15B — LIVE PUBLIC DEPLOYMENT QA
Root NeuroFHIR-QC URL HTTP 200:               PASS
Separate /wish-review/ URL HTTP 200:          PASS
Deployment manifest reachable:                PASS
Frozen public files hash-match deployment:    PASS
Root app and WISH app are distinct:            PASS
Researcher-only leakage scan:                  PASS
Rendered Chromium load:                        PASS
Progressive-disclosure UI/source markers:      PASS
JavaScript page exceptions:                    PASS
Failed same-origin resources:                  PASS
Unexpected network writes on initial load:     PASS
✅ NOTEBOOK 15B LIVE DEPLOYMENT GATE: TRUE

Public participant app:
https://sanghati23.github.io/neurofhir-qc/wish-review/

IMPORTANT:
- P001/P002 remain QA-only IDs.
- Begin real participant allocation with P003.
- Do not use this QA notebook to generate study responses.
